# Catalog Creation

Prepare the supplied HUC8 domains and create the replacement catalog using `configs/params-config.json`. Follow the [rerun runbook](../docs/source/lwi_geometry_rerun.rst). The historical catalog remains unchanged.


In [ ]:
from copy import deepcopy
from functools import reduce
from hashlib import sha256
from pathlib import Path
import json

import geopandas as gpd
from shapely.geometry import mapping

from stormhub.logger import initialize_logger
from stormhub.met.storm_catalog import new_catalog

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

SETTINGS_PATH = REPO_ROOT / "configs" / "params-config.json"
settings = json.loads(SETTINGS_PATH.read_text(encoding="utf-8"))
CATALOG_ROOT = REPO_ROOT / "catalogs"
CATALOG_ID = settings["catalog_id"]
CATALOG_DIR = CATALOG_ROOT / CATALOG_ID
CONFIG_PATH = CATALOG_DIR / "lwi-r3-config.json"
WATERSHED_PATH = REPO_ROOT / settings["watershed"]["geometry_file"]
TRANSPOSITION_REGION_PATH = REPO_ROOT / settings["transposition_region"]["geometry_file"]

if CATALOG_DIR.exists():
    raise FileExistsError(f"Refusing to recreate an existing catalog directory: {CATALOG_DIR}. Use the population notebook to continue an existing generation.")
CATALOG_DIR, WATERSHED_PATH, TRANSPOSITION_REGION_PATH


## Inspect and prepare source geometries

The supplied `.json` files are Esri JSON. Preserve them, verify their pinned bytes, reproject to WGS84, and union all watershed features without simplifying, buffering, or repairing their geometry. StormHub requires one valid Polygon. Preparation fails if that contract cannot be satisfied.


In [ ]:
def prepare_domain(section: dict) -> gpd.GeoDataFrame:
    """Verify source identity and preserve its full footprint as one WGS84 polygon."""
    path = REPO_ROOT / section["geometry_file"]
    if sha256(path.read_bytes()).hexdigest() != section["source_sha256"]:
        raise ValueError(f"Source checksum changed; review the input and update its identity: {path}")
    source = gpd.read_file(path)
    if source.empty or source.crs is None:
        raise ValueError(f"Expected polygon features with a declared CRS: {path}")
    if any(geom is None or geom.is_empty or not geom.is_valid or geom.geom_type not in ("Polygon", "MultiPolygon") for geom in source.geometry):
        raise ValueError(f"Source contains missing, empty, invalid, or non-polygon geometry: {path}")
    projected = source.to_crs("EPSG:4326")
    # Pairwise union preserves all features and avoids the installed Shapely/NumPy bulk-union incompatibility.
    polygon = reduce(lambda left, right: left.union(right), projected.geometry)
    if polygon.geom_type == "MultiPolygon" and len(polygon.geoms) == 1:
        polygon = polygon.geoms[0]
    if polygon.geom_type != "Polygon" or polygon.is_empty or not polygon.is_valid:
        raise ValueError(f"Combined domain must be one valid Polygon; got {polygon.geom_type}: {path}")
    display({"path": section["geometry_file"], "source_features": len(source), "source_crs": str(source.crs), "prepared_crs": "EPSG:4326", "bounds": polygon.bounds, "source_sha256": section["source_sha256"]})
    return gpd.GeoDataFrame(geometry=[polygon], crs="EPSG:4326")

watershed_gdf = prepare_domain(settings["watershed"])
transposition_gdf = prepare_domain(settings["transposition_region"])


In [ ]:
ax = transposition_gdf.boundary.plot(figsize=(8, 8), color="tab:orange", linewidth=2)
watershed_gdf.boundary.plot(ax=ax, color="tab:blue", linewidth=1)
ax.set_title("LWI Region 3 Watershed and Transposition Region")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude");

## Write prepared domains and catalog config

Create a fresh `catalogs/lwi-region3-geometry-v2/` directory. The prepared single-feature GeoJSON files, source identities, and exact run settings are stored there. The runtime config uses local absolute paths because StormHub resolves input paths from the process working directory; the tracked settings use repository-relative paths.


In [ ]:
CATALOG_DIR.mkdir(parents=True, exist_ok=False)
prepared_dir = CATALOG_DIR / "inputs"
prepared_dir.mkdir()
config = deepcopy(settings)
for key, frame in (("watershed", watershed_gdf), ("transposition_region", transposition_gdf)):
    prepared_path = prepared_dir / f"{settings[key]['id']}.geojson"
    feature_collection = {"type": "FeatureCollection", "features": [{"type": "Feature", "properties": {}, "geometry": mapping(frame.geometry.iloc[0])}]}
    payload = (json.dumps(feature_collection, separators=(",", ":")) + "\n").encode("utf-8")
    prepared_path.write_bytes(payload)
    config[key]["source_geometry_file"] = settings[key]["geometry_file"]
    config[key]["geometry_file"] = str(prepared_path.resolve())
    config[key]["prepared_sha256"] = sha256(payload).hexdigest()

(CATALOG_DIR / "params-config.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
CONFIG_PATH.write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")
CONFIG_PATH


## Create base STAC catalog

This creates `catalogs/lwi-region3-geometry-v2/catalog.json` and the `hydro_domains` Items. It accesses the sample AORC S3 dataset to derive the valid transposition region. Run this cell once after inspecting the prepared geometries; it does not populate storm-event collections.


In [ ]:
if (CATALOG_DIR / "catalog.json").exists():
    raise FileExistsError("Base catalog already exists; use the population notebook.")
initialize_logger()
catalog = new_catalog(
    catalog_id=CATALOG_ID,
    config_file=str(CONFIG_PATH),
    local_directory=str(CATALOG_ROOT),
    catalog_description="LWI Region 3 HUC8 geometry revision storm catalog",
)
catalog.spm.catalog_file


In [ ]:
sorted(p.relative_to(CATALOG_DIR) for p in CATALOG_DIR.rglob("*.json"))

## Serve The Catalog

Run this from a PowerShell terminal with the `stormhub` environment active:

```powershell
stormhub-server .\catalogs\lwi-region3-geometry-v2 127.0.0.1 5000
```

Then open `http://localhost:5000/catalog.json` or use the STAC Browser link shown by the directory listing.